# Experiment: 00 Tidy Results

Objective:
- Convert the existing JSON artifacts into flat table-style records for plotting.
- Validate available runs before creating publication figures.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

candidate_dirs = [Path.cwd(), Path.cwd() / "notebooks", *[parent / "notebooks" for parent in Path.cwd().parents]]
for candidate in candidate_dirs:
    if (candidate / "_viz_utils.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError("Could not locate notebooks/_viz_utils.py from current working directory")

import matplotlib.pyplot as plt
import numpy as np

import _viz_utils as vu

plt.style.use("seaborn-v0_8-whitegrid")


In [2]:
all_results = vu.load_all_results()
layer_rows = vu.collect_layer_rows(all_results)
best_rows = vu.collect_best_layer_rows(layer_rows)
cross_rows = vu.collect_cross_rows(all_results)

qwen_math = vu.load_result_json("qwen", "math500")
deepseek_math = vu.load_result_json("deepseek", "math500")
difficulty_rows = vu.collect_difficulty_rows("qwen", qwen_math) + vu.collect_difficulty_rows("deepseek", deepseek_math)
subject_rows = vu.collect_subject_rows("qwen", qwen_math) + vu.collect_subject_rows("deepseek", deepseek_math)

print(f"layer rows: {len(layer_rows)}")
print(f"best-layer rows: {len(best_rows)}")
print(f"cross-transfer rows: {len(cross_rows)}")
print(f"difficulty rows: {len(difficulty_rows)}")
print(f"subject rows: {len(subject_rows)}")


layer rows: 29
best-layer rows: 6
cross-transfer rows: 12
difficulty rows: 16
subject rows: 14


In [3]:
print("Best-layer rows preview:")
for row in sorted(best_rows, key=lambda r: (r["model"], r["dataset"]))[:8]:
    print(
        f"{row['model']:>13} | {row['dataset']:<7} | L{row['layer']:<2} | "
        f"entropy={row['entropy_auc']:.3f} mahal={row['mahal_auc']:.3f} "
        f"combined={row['combined_auc']:.3f} d_raw={row['delta_raw']:+.3f} d_len={row['delta_len_ctrl']:+.3f}"
    )


Best-layer rows preview:
     deepseek | gsm8k   | L7  | entropy=0.728 mahal=0.806 combined=0.835 d_raw=+0.106 d_len=+0.080
     deepseek | math500 | L7  | entropy=0.776 mahal=0.826 combined=0.859 d_raw=+0.083 d_len=+0.044
deepseek_temp | gsm8k   | L7  | entropy=0.699 mahal=0.784 combined=0.822 d_raw=+0.124 d_len=+0.100
         qwen | gsm8k   | L21 | entropy=0.760 mahal=0.690 combined=0.781 d_raw=+0.021 d_len=+0.014
         qwen | math500 | L7  | entropy=0.713 mahal=0.742 combined=0.772 d_raw=+0.059 d_len=+0.027
   qwen_dense | math500 | L8  | entropy=0.713 mahal=0.733 combined=0.760 d_raw=+0.047 d_len=+0.018


In [4]:
export = False
out_dir = vu.results_root() / "tidy"

if export:
    vu.write_csv(out_dir / "layer_rows.csv", layer_rows)
    vu.write_csv(out_dir / "best_rows.csv", best_rows)
    vu.write_csv(out_dir / "cross_rows.csv", cross_rows)
    vu.write_csv(out_dir / "difficulty_rows.csv", difficulty_rows)
    vu.write_csv(out_dir / "subject_rows.csv", subject_rows)
    print(f"Wrote tidy CSV files to: {out_dir}")
else:
    print("Set export=True to write tidy CSV files under results/tidy/")


Set export=True to write tidy CSV files under results/tidy/
